# 01 â€” Prepare Held-Out Test Set

Creates a held-out test set **exclusively from private PERPHECT data** for final model evaluation.
This test set is **never used during training**.

**Data provenance:**
- Only pairs where the phage belongs to the `PERPHECT_private` source are included.
- **Positives** (label=1): Phage-host pairs from `PERPHECT_private` that are not marked as
  negative in `private_interactions`.
- **Negatives** (label=0): Pairs from `PERPHECT_private` with explicit negative interaction
  in `private_interactions` (real negatives only â€” no synthetic generation).

**Output files** (saved in `test_data/`):
- `test_set.csv` â€” string IDs, labels, and sources (human-readable)
- `excluded_pairs.csv` â€” Phage_ID,Host_ID to pass to `train.py --exclude-ids`
- `test_set.npz` â€” precomputed numpy arrays for `02_evaluate_model.ipynb`

**Run this once** before training. The test set is fixed and reusable across experiments.

## 1. Connect to PBI-Scope

In [ ]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path.cwd()))
from pbi import quick_connect
from pbi_adapter import PBIAdapter

In [ ]:
retriever = quick_connect()
adapter = PBIAdapter(retriever)

BACTERIUM_THRESHOLD = 7_000_000
PHAGE_THRESHOLD = 200_000

## 2. Load Private PERPHECT Pairs and Classify

Query phage-host pair IDs where the phage belongs to the `PERPHECT_private` source.
Classify each pair using the `private_interactions` table:
- Pairs with negative interaction type â†’ negatives
- Everything else â†’ positives

In [ ]:
query = """
SELECT DISTINCT pha.Phage_ID, pha.Host_ID
FROM phage_host_associations pha
JOIN fact_phages p ON pha.Phage_ID = p.Phage_ID
WHERE p.Source_DB = 'PERPHECT_private'
ORDER BY MD5(pha.Phage_ID || pha.Host_ID)
"""
all_pairs = retriever.conn.execute(query).fetchdf()
# Filter out pairs where host/phage FASTA is not registered
all_pairs = adapter._filter_pairs_without_sequences(all_pairs)
print(f"Total PERPHECT_private pairs: {len(all_pairs):,}")

positive_pairs, private_negatives = adapter.classify_pairs_by_interaction(all_pairs)
print(f"\nPositive pairs: {len(positive_pairs):,}")
print(f"True negatives (private data): {len(private_negatives):,}")

## 3. Combine and Split

Merge positives and private negatives into a single DataFrame, then split off 15% as the held-out test set.
The split is stratified by class.

In [ ]:
# Combine into single DataFrame with labels
pos_df = positive_pairs.copy()
pos_df["label"] = 1
pos_df["source"] = "positive"

priv_neg_df = private_negatives.copy()
priv_neg_df["label"] = 0
priv_neg_df["source"] = "private_data"

all_data = pd.concat([pos_df, priv_neg_df], ignore_index=True)
print(f"Total pairs: {len(all_data):,}")
print(f"  Positives: {int(all_data['label'].sum()):,}")
print(f"  Negatives: {int(len(all_data) - all_data['label'].sum()):,}")

In [ ]:
# Stratified split: 85% train, 15% test
train_df, test_df = train_test_split(
    all_data, stratify=all_data["label"], test_size=0.15, shuffle=True, random_state=42
)

print(f"Train: {len(train_df):,} pairs")
print(f"Test:  {len(test_df):,} pairs")
print(f"\nTest set class breakdown:")
print(test_df["source"].value_counts().to_string())

## 4. Fetch and Encode Test Sequences

In [ ]:
print(f"Fetching sequences for {len(test_df)} test pairs...")

bacteria_seqs = []
phage_seqs = []
bacteria_raw = []
phage_raw = []
valid_mask = []

for i, row in test_df.iterrows():
    bseq = adapter._fetch_host_sequence(row["Host_ID"])
    pseq = adapter._fetch_phage_sequence(row["Phage_ID"])
    if bseq is not None and pseq is not None:
        bacteria_seqs.append(adapter._pad_and_encode(bseq, BACTERIUM_THRESHOLD))
        phage_seqs.append(adapter._pad_and_encode(pseq, PHAGE_THRESHOLD))
        bacteria_raw.append(bseq)
        phage_raw.append(pseq)
        valid_mask.append(True)
    else:
        valid_mask.append(False)

valid_mask = np.array(valid_mask)
n_dropped = len(valid_mask) - valid_mask.sum()
if n_dropped > 0:
    print(f"Dropped {n_dropped} pairs (missing/too-short sequences)")

bacteria_arr = np.stack(bacteria_seqs)
phage_arr = np.stack(phage_seqs)
test_df = test_df[valid_mask].reset_index(drop=True)

print(f"Encoded: bacteria={bacteria_arr.shape}, phage={phage_arr.shape}")

## 5. Save Test Set

In [ ]:
out_dir = Path.cwd() / "test_data"
out_dir.mkdir(exist_ok=True)

# Save test set CSV
test_df.to_csv(out_dir / "test_set.csv", index=False)
print(f"Saved test_set.csv ({len(test_df)} rows)")

# Save excluded pair IDs (for train.py --exclude-ids to prevent data leakage)
test_df[["Phage_ID", "Host_ID"]].to_csv(out_dir / "excluded_pairs.csv", index=False)
print(f"Saved excluded_pairs.csv ({len(test_df)} rows)")

# Save encoded sequences (for 02_evaluate_model.ipynb â€” no DB needed)
np.savez(
    out_dir / "test_set.npz",
    bacteria=bacteria_arr,
    phage=phage_arr,
    labels=test_df["label"].values.astype(np.float32),
    sources=test_df["source"].values,
)
print(f"Saved test_set.npz")

## 6. Export Raw Sequences

In [ ]:
export_df = test_df[["Phage_ID", "Host_ID", "label", "source"]].copy()
export_df["bacteria_sequence"] = bacteria_raw
export_df["phage_sequence"] = phage_raw

export_df.to_csv(out_dir / "export_test_set.csv", index=False)
print(f"Saved export_test_set.csv ({len(export_df)} rows, with raw nucleotide sequences)")

## Done

The held-out test set (private PERPHECT data only) is ready. Files saved in `test_data/`:

| File | Contents |
|------|----------|
| `test_set.csv` | Phage_ID, Host_ID, label, source |
| `excluded_pairs.csv` | Phage_ID, Host_ID (for `--exclude-ids` in train.py) |
| `test_set.npz` | `bacteria` (N,T,4), `phage` (N,T,4), `labels` (N,), `sources` (N,) |
| `export_test_set.csv` | Phage_ID, Host_ID, label, source, bacteria_sequence, phage_sequence |

**Next steps:**
1. Train a model (excluding test pairs):
   ```bash
   python train.py --config config.yaml --exclude-ids test_data/excluded_pairs.csv
   ```
2. Evaluate: open `02_evaluate_model.ipynb`